# Documentation Gap Audit Notebook

Portable documentation-coverage audit for a new Snowflake environment. Run top to bottom. It
surfaces two different kinds of documentation gap:

1. **Object-level documentation inside Snowflake itself** — `COMMENT` fields on databases,
   schemas, tables/views, warehouses, and resource monitors. This is queryable directly, so
   Sections 1–4 are SQL.
2. **Process-level documentation** — runbooks, onboarding guides, incident response,
   disaster recovery, and change management procedures. Snowflake has no view for "does a
   runbook exist," so Section 5 is a checklist to work through with the team and track
   completion against, not a query.

### Prerequisites
- A role with `IMPORTED PRIVILEGES` on the `SNOWFLAKE` database (or `ACCOUNTADMIN`).
- Nothing here is destructive — every SQL cell is a `SELECT`.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET min_table_rows_to_flag = 1000;   -- ignore trivially small/empty tables when ranking gaps by importance

SELECT $min_table_rows_to_flag AS min_table_rows_to_flag;


## 1. Database & schema comment coverage

The highest-level objects are also the cheapest to document and the most valuable to a new
team member trying to orient themselves — a missing comment here is a bigger gap per-object
than a missing comment on one column.


In [ ]:
-- Databases and schemas with no comment
SELECT 'DATABASE' AS object_type, database_name AS object_name, comment, created
FROM snowflake.account_usage.databases
WHERE deleted IS NULL
  AND (comment IS NULL OR TRIM(comment) = '')

UNION ALL

SELECT 'SCHEMA', catalog_name || '.' || schema_name, comment, created
FROM snowflake.account_usage.schemata
WHERE deleted IS NULL
  AND schema_name NOT IN ('INFORMATION_SCHEMA')
  AND (comment IS NULL OR TRIM(comment) = '')

ORDER BY object_type, object_name;


## 2. Table & view comment coverage

Ranked so the highest-row-count undocumented tables surface first — those are the ones most
likely to be depended on by others and most costly to leave unexplained.


In [ ]:
-- Undocumented tables and views, ranked by size/importance
SELECT
    table_catalog || '.' || table_schema || '.' || table_name AS full_name,
    table_type,
    row_count,
    ROUND(bytes / POWER(1024,3), 2) AS gb,
    table_owner,
    last_altered
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
  AND (comment IS NULL OR TRIM(comment) = '')
  AND (row_count IS NULL OR row_count >= $min_table_rows_to_flag)
ORDER BY row_count DESC NULLS LAST
LIMIT 200;


In [ ]:
-- Coverage percentage by schema, to prioritize which schema to tackle first
SELECT
    table_catalog,
    table_schema,
    COUNT(*)                                                        AS total_objects,
    SUM(IFF(comment IS NULL OR TRIM(comment) = '', 1, 0))           AS undocumented_objects,
    ROUND(100.0 * SUM(IFF(comment IS NULL OR TRIM(comment) = '', 1, 0)) / COUNT(*), 1) AS pct_undocumented
FROM snowflake.account_usage.tables
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
GROUP BY table_catalog, table_schema
ORDER BY pct_undocumented DESC, total_objects DESC;


## 3. Column-level documentation coverage

Column-by-column review usually isn't worth doing account-wide — this rolls it up to one row
per table so you can see *which* tables have poor column documentation without drowning in
individual column rows, then drill into a specific table's columns only when it's worth it.


In [ ]:
-- Column comment coverage, rolled up per table
SELECT
    table_catalog,
    table_schema,
    table_name,
    COUNT(*)                                                     AS total_columns,
    SUM(IFF(comment IS NULL OR TRIM(comment) = '', 1, 0))        AS undocumented_columns,
    ROUND(100.0 * SUM(IFF(comment IS NULL OR TRIM(comment) = '', 1, 0)) / COUNT(*), 1) AS pct_undocumented
FROM snowflake.account_usage.columns
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
GROUP BY table_catalog, table_schema, table_name
HAVING pct_undocumented > 0
ORDER BY total_columns DESC
LIMIT 200;


## 4. Warehouse & resource monitor documentation

Small in count but high-value to document — anyone troubleshooting a cost or performance
issue at 2am benefits from a warehouse comment that says what it's for and who owns it.


In [ ]:
-- Current warehouse definitions (comment field isn't in ACCOUNT_USAGE, so this uses SHOW)
SHOW WAREHOUSES;


In [ ]:
-- Warehouses with no comment
SELECT "name" AS warehouse_name, "size", "owner", "comment"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE "comment" IS NULL OR TRIM("comment") = ''
ORDER BY warehouse_name;


In [ ]:
-- Resource monitors and whether they're actually assigned to anything
SHOW RESOURCE MONITORS;


## 5. Process/runbook documentation checklist

Not a query — Snowflake has no view for "does a disaster recovery runbook exist." Work
through this with the team and track status directly in this cell (or copy it into whatever
the team's actual documentation tracker is once one exists). Categories are pulled directly
from the JD's stated documentation responsibilities.

| Category | Exists? | Location | Last Updated | Owner | Notes |
|---|---|---|---|---|---|
| Incident response runbook | | | | | |
| Disaster recovery / failover procedure | | | | | |
| New-hire / role onboarding guide (Platform team) | | | | | |
| Change management process (what requires review, by whom) | | | | | |
| Access request & approval process | | | | | |
| Service account provisioning/deprovisioning procedure | | | | | |
| RBAC model documentation (role hierarchy, naming convention) | | | | | |
| Data retention & archival policy | | | | | |
| Cost/consumption monitoring & alerting procedure | | | | | |
| Monthly release-note review process | | | | | |
| Architecture diagrams (current-state) | | | | | |
| Backup/replication configuration reference | | | | | |

**Suggested first pass:** mark each row Exists = Yes/No/Partial, then prioritize "No" rows
that map to something already flagged as high-risk elsewhere (e.g., no documented service
account deprovisioning procedure *and* the service-account notebook found dormant accounts
still enabled — that pairing is a strong argument for prioritizing that runbook first).


### Notes, caveats, and next steps

- **A `COMMENT` isn't automatically good documentation.** A table with `comment = 'temp'` will
  show as "documented" here even though it isn't useful — treat the coverage percentages as a
  gap-finder, not a quality score. Spot-check a sample of "documented" objects too.
- **Column-level documentation is usually not worth doing exhaustively.** Prioritize columns
  on tables that other teams query directly (dashboards, reporting marts) over internal
  staging/intermediate tables.
- **Pair Section 4 with the RBAC audit notebook.** A warehouse with no comment and no clear
  owner is often the same warehouse that turns up as a cost or right-sizing question later —
  documenting ownership tends to also surface who to talk to about usage patterns.
